In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data_train = pd.read_csv('/home/bigbang/machinelearning_algorithms/nn_train.csv')
X_train = data_train.iloc[:, 1:-2].values  
y_train_binary = data_train.iloc[:, -2].values 
y_train_binary = y_train_binary.reshape(-1,1)

data_test = pd.read_csv('/home/bigbang/machinelearning_algorithms/nn_test.csv')
X_test = data_test.iloc[:, 1:].values 

# Normalize training data using Min-Max Normalization
def min_max_normalize(X):
    X_min = np.min(X, axis=0)
    X_max = np.max(X, axis=0)
    return (X - X_min) / (X_max - X_min)

# Normalize training data
X_train = min_max_normalize(X_train)
X_test = min_max_normalize(X_test)


print("Test data after Min-Max normalization:")
print(X_test[:1])  

print("Training data (first 5 rows):")
print(X_train[:1])
print(y_train_binary[:1])

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
#Neural Network hyperparameters 
input_layer_size = X_train.shape[1]
hidden_layer_size = 64 
output_layer_size = 1
alpha = 0.01
epochs = 1000

np.random.seed(42)
W1 = np.random.randn(input_layer_size, hidden_layer_size)
b1 = np.zeros((1, hidden_layer_size))
W2 = np.random.randn(hidden_layer_size, output_layer_size)
b2 = np.zeros((1, output_layer_size))

def sigmoid(z):
    return 1/(1 + np.exp(-z))

def sigmoid_derivative(a):
    return a * (1-a) 

def forward_propagation(X):
    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1) 
    Z2 = np.dot(A1, W2) + b2 
    A2 = sigmoid(Z2)
    return Z1, A1, Z2, A2

def back_propagation(X , y , Z1 , A1, A2):
    m = X.shape[0]
    dZ2 = A2 - y.reshape(-1,1) 
    dW2 = np.dot(A1.T, dZ2) / m
    db2 = np.sum(dZ2, axis = 0)/m

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * sigmoid_derivative(A1)
    dW1 = np.dot(X.T , dZ1)/m 
    db1 = np.sum(dZ1, axis = 0)/m 

    return dW2, dW1, db1 , db2 



In [ ]:
loss_history = []
for i in range(epochs):
    Z1, A1, Z2 , A2 = forward_propagation(X_train)
    dW2,dW1, db1, db2 = back_propagation(X_train,y_train_binary, Z1, A1, A2) 

    W1 -= alpha*dW1
    W2 -= alpha*dW2 
    b1 -= alpha*db1
    b2 -= alpha*db2 

    loss = -np.mean(y_train_binary * np.log(A2) + (1 - y_train_binary) * np.log(1 - A2))
    loss_history.append(loss) 

    if i % 100 == 0:
        print(f"Epoch: {i}, Loss: {loss:.4f}")

plt.plot(loss_history)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

def predict(X):
    Z1,A1,Z2,A2 = forward_propagation(X)
    return(A2 > 0.5).astype(int)

print(A2.shape)
print(y_train_binary.shape)
y_pred_train = predict(X_train)
y_pred_test = predict(X_test)
print(f"prediction of training set: {y_pred_train[:5]}")
print(f"prediction for test data: {y_pred_test}")
